# Inflation in Italy

AIM: show which datasets are available for inflation in Italy, by ISTAT.

In [3]:
import warnings 
from istatapi import discovery, retrieval
import requests

warnings.filterwarnings('ignore')
requests.urllib3.disable_warnings()

! pip show istatapi

Name: istatapi
Version: 1.0.0
Summary: Python API for ISTAT (The Italian National Institute of Statistics)
Home-page: https://github.com/Attol8/istatapi
Author: Jacopo Attolini
Author-email: jacopoattolini@gmail.com
License: Apache Software License 2.0
Location: /Users/danieleongari/opt/anaconda3/envs/py312/lib/python3.12/site-packages
Requires: fastcore, matplotlib, pandas, pyopenssl, requests
Required-by: 


In [4]:
all_datasets = discovery.search_dataset("")
display(all_datasets)

,df_id,version,df_description,df_structure_id
0,101_1015,1.3,Crops,DCSP_COLTIVAZIONI
1,101_1030,1.0,"PDO, PGI and TSG quality products",DCSP_DOPIGP
2,101_1033,1.0,slaughtering,DCSP_MACELLAZIONI
3,101_1039,1.2,Agritourism - municipalities,DCSP_AGRITURISMO_COM
4,101_1077,1.0,"PDO, PGI and TSG products: operators - munici...",DCSP_DOPIGP_COM
...,...,...,...,...
504,97_953,1.1,Environmental protection expenditure,DCCN_SPESAPROTAMB
505,98_1066,1.0,Productivity measures - Accounts in the 2014 v...,DCCN_PRODUTTIVITA_B14
506,98_1067,1.0,Productivity measures - Accounts in the 2011 v...,DCCN_PRODUTTIVITA_B11
507,98_197,1.3,Productivity measures,DCCN_PRODUTTIVITA


In [15]:
selected_datasets = (
    all_datasets
    .query('df_description.str.contains("hicp|nic|foi", case=False)')
    .query('~df_description.str.contains("municipal|polytechnic|communic", case=False)')
    .sort_values('df_structure_id', ascending=True)
)

count=0
for _, row in selected_datasets.iterrows():
    print(f'{count+1:2d} - v{row["version"]} {row["df_structure_id"]}  - {row["df_description"]}')
    count+=1

 1 - v1.1 DCSP_FOI1  - Foi - monthly data until 2010 (base 1995)
 2 - v1.0 DCSP_FOI1B2010  - Foi - monthly data from 2011 to 2015 (base 2010)
 3 - v1.1 DCSP_FOI1B2015  - Foi - monthly data from 2016 (base 2015)
 4 - v1.1 DCSP_FOI2  - Foi - annual data until 2010 (base 1995)
 5 - v1.0 DCSP_FOI2B2010  - Foi - annual data from 2011 to 2015  (base 2010)
 6 - v1.0 DCSP_FOI2B2015  - Foi - annual data from 2016 (base 2015)
 7 - v1.0 DCSP_FOI3B2010  - FOI - Weights from 2011 to 2015
 8 - v1.1 DCSP_FOI3B2015  - FOI - Weights from 2016 onwards
 9 - v1.0 DCSP_IPCA1  - Hicp - monthly and quarterly data from 2001 until 2015(base 2005)
10 - v1.4 DCSP_IPCA1B2015  - Hicp - monthly data from 2001
11 - v1.1 DCSP_IPCA2  - Hicp - annual data from 2001 until 2015 (base 2005)
12 - v1.2 DCSP_IPCA2B2015  - Hicp - annual data from 2001 (base 2015)
13 - v1.2 DCSP_IPCA3  - Hicp - weights  from 2001
14 - v1.2 DCSP_IPCATC1  - Hicp - at constant tax rates monthly data (base 2005)
15 - v1.4 DCSP_IPCATC1B2015  - Hicp

In [18]:
interesting_dss = ["DCSP_IPCA1B2015", "DCSP_FOI1B2015", "DCSP_NICUNOB", "DCSP_NICUNOBB2010", "DCSP_NIC1B2015"] 

for ds_structure_id in interesting_dss:
    print(f"\n############ {ds_structure_id} ################################################################")
    print("Description:", all_datasets.query(f'df_structure_id == "{ds_structure_id}"').iloc[0].df_description)
    print(f"Website: http://dati.istat.it/Index.aspx?DataSetCode={ds_structure_id}")
    try:
        ds = discovery.DataSet(dataflow_identifier=ds_structure_id)
        ds_info = ds.dimensions_info()
        display(ds_info)
        for col in ds_info.dimension:
            print(f" >> {col}")
            display(ds.get_dimension_values(col).sort_values("values_ids").reset_index(drop=True))
            
    except Exception as e:
        print("Error:", e)
        continue


############ DCSP_IPCA1B2015 ################################################################
Description: Hicp - monthly data from 2001
Website: http://dati.istat.it/Index.aspx?DataSetCode=DCSP_IPCA1B2015


,dimension,dimension_ID,description
0,FREQ,CL_FREQ,Frequency
1,MISURA1,CL_MISURA1,MISURA1
2,IND_TYPE,CL_TIPO_DATO2,Data type 2
3,COICOP_IPCA,CL_COICOP_2015,Coicop 2015
4,ITTER107,CL_ITTER107,Territory


 >> FREQ


,values_ids,values_description
0,A,annual
1,M,monthly


 >> MISURA1


...

,values_ids,values_description
0,IT,Italy
1,ITC,Nord-ovest
2,ITC1,Piemonte
3,ITC11,Torino
4,ITC12,Vercelli
...,...,...
122,ITG25,Sassari
123,ITG26,Nuoro
124,ITG27,Cagliari
125,ITG28,Oristano


 >> MISURA1


,values_ids,values_description
0,4,index number
1,6,percentage changes on the previous period
2,7,percentage changes on the same period of the p...


 >> IND_TYPE


,values_ids,values_description
0,39,consumer price index for the whole nation (bas...
1,9,consumer price index for the whole nation (bas...
